# 02 EDA & KPI Logic

This notebook explains the KPI logic behind the dashboard and provides a compact exploration of readiness, risk exposure, supplier variance, and mitigation progress.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import plotly.express as px

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from calculate_kpis import portfolio_kpis, REFERENCE_DATE
from risk_scoring import calculate_readiness_scores, risk_exposure_by_workstream
from utils import load_all

data = load_all(processed=True)
kpis = portfolio_kpis(data['collections'], data['milestones'], data['risks'], data['actions'], data['purchase_orders'], data['suppliers'], REFERENCE_DATE)
kpis

## Readiness Score

The readiness score starts at 100 and subtracts for open critical risks, blocked milestones, overdue actions, average delay, poor supplier reliability, and unresolved issues close to launch. It adds modest credit for completed milestones and closed mitigations. The score is clipped between 0 and 100 and grouped into Ready, Watch, At Risk, and Critical bands.

In [ ]:
readiness = calculate_readiness_scores(data['collections'], data['milestones'], data['risks'], data['actions'], data['purchase_orders'], data['suppliers'], REFERENCE_DATE)
readiness.merge(data['collections'][['collection_id', 'collection_name', 'brand_name', 'market', 'launch_date']], on='collection_id').sort_values('readiness_score').head(10)

## Risk Exposure

Risk score is probability x impact. Severity bands make the risk table easier to govern: Low, Medium, High, and Critical. Open risk exposure by workstream shows where leadership attention is most needed.

In [ ]:
risk_exposure_by_workstream(data['risks']).head(10)

## Supplier and Delay Patterns

Supplier reliability influences lead-time variance, incident probability, and delay exposure. This view highlights the suppliers that would deserve a sourcing review before final launch approval.

In [ ]:
supplier_perf = data['purchase_orders'].groupby(['supplier_name', 'country'], as_index=False).agg(
    avg_delay=('delay_days', 'mean'),
    lead_time_variance=('lead_time_variance_days', 'mean'),
    orders=('po_id', 'count')
).sort_values('avg_delay', ascending=False)
supplier_perf.head(15)

## Example Insight

A practical project review would start with the lowest-readiness collections, then check whether blocked milestones are linked to supplier, quality, logistics, or content readiness. Overdue mitigation actions are the fastest operational lever because they identify named owners and due dates.